In [0]:
from datetime import datetime
from pyspark.sql import Row

print("Körning:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

kontroller = [
    ("bronze", "nyreg_bilar_raw",  "dbjahan.bronze_data.nyreg_bilar_raw",  "region, drivmedel, tid"),
    ("bronze", "bilar_trafik_raw", "dbjahan.bronze_data.bilar_trafik_raw", "region, agarkategori, tid"),
    ("bronze", "folkmangd_raw",    "dbjahan.bronze_data.folkmangd_raw",    "Region, Alder, Civilstand, Kon, Tid"),
    ("bronze", "elpris_raw",       "dbjahan.bronze_data.elpris_raw",       "elomrade, tid_start"),
    ("silver", "nyreg_bilar",      "dbjahan.silver_data.nyreg_bilar",      "kommun_kod, drivmedel_kod, ar, manad"),
    ("silver", "bilar_trafik",     "dbjahan.silver_data.bilar_trafik",     "kommun_kod, ar"),
    ("silver", "folkmangd",        "dbjahan.silver_data.folkmangd",        "kommun_kod, ar"),
    ("silver", "elpris",           "dbjahan.silver_data.elpris",           "elomrade, ar"),
    ("silver", "kommun_elomrade",  "dbjahan.silver_data.kommun_elomrade",  "kommun_kod"),
    ("gold",   "dim_kommun",       "dbjahan.gold_data.dim_kommun",         "kommun_kod"),
    ("gold",   "fakt_laddplats",   "dbjahan.gold_data.fakt_laddplats",     "kommun_kod"),
    ("gold",   "topp_50_kommuner", "dbjahan.gold_data.topp_50_kommuner",   "kommun_kod"),
]

rader = []
for lager, namn, tabell, nyckel in kontroller:
    antal = spark.sql(f"SELECT COUNT(*) AS n FROM {tabell}").first().n
    dubletter = spark.sql(f"""
        SELECT COUNT(*) AS n FROM (
          SELECT {nyckel} FROM {tabell}
          GROUP BY {nyckel}
          HAVING COUNT(*) > 1
        )
    """).first().n
    rader.append(Row(lager=lager, tabell=namn, antal_rader=antal, dubblettnycklar=dubletter))

display(spark.createDataFrame(rader))

Körning: 2026-10-07 20:48:44


lager,tabell,antal_rader,dubblettnycklar
bronze,nyreg_bilar_raw,143640,0
bronze,bilar_trafik_raw,13230,0
bronze,folkmangd_raw,12480,0
bronze,elpris_raw,61548,0
silver,nyreg_bilar,132696,0
silver,bilar_trafik,1746,0
silver,folkmangd,1450,0
silver,elpris,4,0
silver,kommun_elomrade,290,0
gold,dim_kommun,290,0
